# Step 3 - Eksekusi Final: Agregasi, Pembersihan, Split `data/processed/`

Eksekusi sesuai konfirmasi di `step2_prompt.md`:
1. **Metode agregasi:** Majority Voting + Safety-First (`v/n >= 0.5`) untuk **seluruh 6 label**
2. **Tie-break flag:** `tie_break_applied_<label>` (bool) per label per teks
3. **Pembersihan:** modus konten per `text_id`, pemulihan teks kosong, normalisasi topic
4. **Split:** Group-aware Multi-label Iterative Stratification (group_id dari konten dinormalisasi) -> 70/15/15
5. Output ke `data/processed/`

> Catatan rekonsiliasi (lihat `resumedata.md` bagian Rekonsiliasi): angka any-positive di file ini = 4.689, bukan 4.236 seperti "sesi sebelumnya"; majority = 2.404 (cocok). Tidak mempengaruhi metode final.

In [1]:
# Yang perlu diganti hanya 2 baris di bawah ini.
# data: folder yang berisi 2 file jsonl dataset mentah
# folder_output: hasil agregasi dan split akan disimpan di sini

data = r'C:\SEMESTER 5\Project Sistem Cerdas\Projek\Dataset'
folder_output = r'C:\SEMESTER 5\Project Sistem Cerdas\Projek\data\processed'

# Kode di bawah baris ini tidak perlu diubah.
import os, json, re, hashlib
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from skmultilearn.model_selection import IterativeStratification

if not os.path.isdir(data):
    raise FileNotFoundError(
        f"Folder tidak ketemu: {data}. "
        "Perbaiki isi variabel data di baris paling atas sel ini.")

SEED = 42
np.random.seed(SEED)

DATASET_DIR = data
OUT_DIR = folder_output
os.makedirs(OUT_DIR, exist_ok=True)

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

annotated = load_jsonl(os.path.join(DATASET_DIR, 'indotoxic2024_annotated_data-3.jsonl'))
annotators = load_jsonl(os.path.join(DATASET_DIR, 'indotoxic2024_annotator_data.jsonl'))
print(f'Path dataset : {DATASET_DIR}')
print(f'Path output  : {OUT_DIR}')
print(f'Baris anotasi   : {len(annotated)}')
print(f'Profil anotator : {len(annotators)}')

LABEL_COLS = ['toxicity', 'profanity_obscenity', 'threat_incitement_to_violence',
              'insults', 'identity_attack', 'sexually_explicit']


Path dataset : C:\SEMESTER 5\Project Sistem Cerdas\Projek\Dataset
Path output  : C:\SEMESTER 5\Project Sistem Cerdas\Projek\data\processed
Baris anotasi   : 43692
Profil anotator : 19


## 1. Pembersihan: modus konten, pemulihan teks kosong, normalisasi topic

In [2]:
texts = defaultdict(list)
for o in annotated:
    texts[o['text_id']].append(o)

clean_rows = []
drop_log = Counter()
recovered = []

for tid, rows in texts.items():
    # Kanonik = modus konten (strip)
    canon = Counter(r['text'].strip() for r in rows).most_common(1)[0][0]
    kept = [r for r in rows if r['text'].strip() == canon]
    drop_log['baris_konten_minoritas'] += len(rows) - len(kept)
    if canon == '':
        drop_log['unit_kosong_total'] += 1
        drop_log['baris_unit_kosong'] += len(rows)
        continue
    if any(not r['text'].strip() for r in rows):
        recovered.append(tid)
    # Anotator ganda (baris redundan dari anotator sama): tetap dipertahankan
    # sebagai vote terpisah (asumsi: penilaian independen) - didokumentasikan.
    for r in kept:
        clean_rows.append({
            'text_id': tid,
            'text': canon,
            'initial_paragraph': r['initial_paragraph'],
            'topic_raw': r['topic'].strip(),
            'topic': 'UNKNOWN' if r['topic'].strip() in ('1', 'UNKNOWN') else r['topic'].strip(),
            'annotator_id': r['annotator_id'],
            **{c: r[c] for c in LABEL_COLS + ['is_noise_or_spam_text', 'related_to_election_2024']},
        })

df = pd.DataFrame(clean_rows)
df['topic'] = df['topic'].apply(lambda s: re.sub(r'\s*,\s*', ', ', s))
print('=== LOG PEMBERSIHAN ===')
for k, v in sorted(drop_log.items()):
    print(f'  {k:28s}: {v}')
print(f'  unit TERPULIHKAN (teks kosong): {recovered}')
print(f'\nBaris valid setelah bersih : {len(df)}')
print(f'Teks unik (text_id)        : {df.text_id.nunique()}')
print(f'topic "1" tersisa          : {(df.topic == "1").sum()}')
print(f'topic UNKNOWN              : {(df.topic == "UNKNOWN").sum()}')

=== LOG PEMBERSIHAN ===
  baris_konten_minoritas      : 171
  baris_unit_kosong           : 1
  unit_kosong_total           : 1
  unit TERPULIHKAN (teks kosong): ['103-150']

Baris valid setelah bersih : 43520
Teks unik (text_id)        : 28448
topic "1" tersisa          : 0
topic UNKNOWN              : 3903


## 2. Agregasi per teks: majority + safety-first (`v/n >= 0.5`), seluruh 6 label

In [3]:
agg_rows = []
for tid, g in df.groupby('text_id', sort=False):
    n = len(g)
    rec = {
        'text_id': tid,
        'text': g['text'].iloc[0],
        'initial_paragraph': g['initial_paragraph'].iloc[0],
        'topic': g['topic'].iloc[0],
        'n_annotators': n,
    }
    for c in LABEL_COLS:
        v = int(g[c].sum())
        rec[c] = int(v / n >= 0.5)              # final label (safety-first)
        rec['agreement_' + c] = v / n           # metadata agreement
        rec['tie_break_applied_' + c] = bool(n % 2 == 0 and v == n / 2)
    rec['is_noise_or_spam_text'] = int(g['is_noise_or_spam_text'].sum() / n >= 0.5)
    rec['related_to_election_2024'] = int(g['related_to_election_2024'].sum() / n >= 0.5)
    agg_rows.append(rec)

dft = pd.DataFrame(agg_rows)
print(f'Unit hasil agregasi: {len(dft)}')
print('\nDistribusi label final (safety-first):')
for c in LABEL_COLS:
    n_pos = dft[c].sum()
    n_tie = dft['tie_break_applied_' + c].sum()
    print(f'  {c:30s} : {n_pos:>6,} positif ({n_pos/len(dft):5.1%}) | tie-break: {n_tie:>5,}')
print('\nDistribusi n_annotators:')
print(dft.n_annotators.value_counts().sort_index().to_string())

Unit hasil agregasi: 28448

Distribusi label final (safety-first):
  toxicity                       :  4,414 positif (15.5%) | tie-break: 1,996
  profanity_obscenity            :    904 positif ( 3.2%) | tie-break:   519
  threat_incitement_to_violence  :  1,068 positif ( 3.8%) | tie-break:   945
  insults                        :  2,260 positif ( 7.9%) | tie-break: 1,376
  identity_attack                :  2,186 positif ( 7.7%) | tie-break: 1,242
  sexually_explicit              :    151 positif ( 0.5%) | tie-break:    95

Distribusi n_annotators:
n_annotators
1     18169
2      9759
3        70
5         1
10        7
11       95
12        8
13      339


## 3. `group_id` - pengelompokan konten identik/near-duplicate

Kunci grup = konten dinormalisasi (lowercase, whitespace & tanda kutip dibersihkan). Konten kosong tidak ada lagi. Semua `text_id` yang kontennya sama -> satu `group_id` -> dijamin masuk split yang sama.

In [4]:
def norm_text(s):
    s = re.sub(r'\s+', ' ', s).strip().strip('"').strip("'").strip().lower()
    return s

dft['group_key'] = dft['text'].map(norm_text)
uniq_keys = {k: i for i, k in enumerate(sorted(dft['group_key'].unique()))}
dft['group_id'] = dft['group_key'].map(uniq_keys)

gsz = dft.groupby('group_id').size()
print(f'Unit (text_id) : {len(dft)}')
print(f'Grup unik      : {len(uniq_keys)}')
print(f'Unit dalam grup multi-anggota: {(gsz > 1).sum() and int(gsz[gsz > 1].sum())} '
      f'({gsz[gsz > 1].sum()/len(dft):.1%} dari total)')
print('Distribusi ukuran grup (top 8):')
print(gsz.value_counts().sort_index().head(8).to_string())

Unit (text_id) : 28448
Grup unik      : 26157
Unit dalam grup multi-anggota: 4425 (15.6% dari total)
Distribusi ukuran grup (top 8):
1    24023
2     2003
3      114
4        8
5        9


## 4. Split: group-aware multi-label iterative stratification (70/15/15)

Label yang diseimbangkan = **label agregat grup** (OR/Max dari anggota grup). Hasil split grup kemudian di-*re-map* ke semua unit anggotanya.

In [5]:
gdf = dft.groupby('group_id', sort=False)[LABEL_COLS].max()
gdf['n_units'] = dft.groupby('group_id', sort=False).size()
Y = gdf[LABEL_COLS].values.astype(int)
X_idx = np.arange(len(gdf)).reshape(-1, 1)

strat = IterativeStratification(n_splits=3, order=1,
                                sample_distribution_per_fold=[0.70, 0.15, 0.15])
fold_of_group = np.empty(len(gdf), dtype=int)
for fold, (_, test_idx) in enumerate(strat.split(X_idx, Y)):
    fold_of_group[test_idx] = fold

SPLIT_NAMES = ['train', 'val', 'test']
gdf['split'] = [SPLIT_NAMES[f] for f in fold_of_group]
dft = dft.merge(gdf['split'], left_on='group_id', right_index=True, how='left')

print('Unit per split :', dict(dft.split.value_counts().reindex(SPLIT_NAMES)))
print('Grup per split :', dict(gdf.split.value_counts().reindex(SPLIT_NAMES)))
print('\nLeakage check: grup lintas split =',
      int((dft.groupby('group_id')['split'].nunique() > 1).sum()))

Unit per split : {'train': 19986, 'val': 4229, 'test': 4233}
Grup per split : {'train': 18309, 'val': 3924, 'test': 3924}

Leakage check: grup lintas split = 0


In [6]:
print('Distribusi label (unit-level) per split:')
rows = []
for s in SPLIT_NAMES:
    sub = dft[dft.split == s]
    row = {'split': s, 'n_unit': len(sub)}
    for c in LABEL_COLS:
        row[c] = f"{sub[c].mean():5.1%}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

print('\nDistribusi label (group-level, agregat OR) per split:')
rows = []
for s in SPLIT_NAMES:
    sub = gdf[gdf.split == s]
    row = {'split': s, 'n_grup': len(sub)}
    for c in LABEL_COLS:
        row[c] = f"{sub[c].mean():5.1%}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

Distribusi label (unit-level) per split:
split  n_unit toxicity profanity_obscenity threat_incitement_to_violence insults identity_attack sexually_explicit
train   19986    15.3%                2.7%                          3.7%    7.8%            7.6%              0.6%
  val    4229    16.0%                4.3%                          3.8%    8.4%            7.9%              0.4%
 test    4233    15.8%                4.3%                          3.8%    8.4%            7.8%              0.3%

Distribusi label (group-level, agregat OR) per split:
split  n_grup toxicity profanity_obscenity threat_incitement_to_violence insults identity_attack sexually_explicit
train   18309    16.2%                2.9%                          4.1%    8.3%            8.2%              0.7%
  val    3924    16.2%                4.4%                          4.1%    8.6%            8.2%              0.4%
 test    3924    16.2%                4.4%                          4.1%    8.5%            8.2%   

## 5. Simpan ke `data/processed/`

| File | Isi |
|---|---|
| `train.jsonl` / `val.jsonl` / `test.jsonl` | 1 baris = 1 teks: konten + label final + metadata agreement lengkap |
| `agreement_metadata.jsonl` | subset kolom metadata agreement per teks |
| `splits_summary.json` | statistik pipeline (angka audit) |

In [7]:
META_COLS = (['text_id', 'group_id', 'topic', 'initial_paragraph', 'n_annotators', 'split']
             + ['agreement_' + c for c in LABEL_COLS]
             + ['tie_break_applied_' + c for c in LABEL_COLS]
             + ['is_noise_or_spam_text', 'related_to_election_2024'])

def dump_jsonl(df_, path):
    with open(path, 'w', encoding='utf-8') as f:
        for rec in df_.to_dict(orient='records'):
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

for s in SPLIT_NAMES:
    dump_jsonl(dft[dft.split == s], os.path.join(OUT_DIR, s + '.jsonl'))
dump_jsonl(dft[META_COLS], os.path.join(OUT_DIR, 'agreement_metadata.jsonl'))

summary = {
    'seed': SEED,
    'aggregation_method': 'majority_safety_first (v/n >= 0.5), semua 6 label',
    'rows_input': len(annotated),
    'rows_clean': int(len(df)),
    'rows_dropped_minority_content': int(drop_log['baris_konten_minoritas']),
    'units_empty_dropped': int(drop_log['unit_kosong_total']),
    'units_text_recovered': sorted(recovered),
    'units_final': int(len(dft)),
    'groups_final': int(gdf.index.nunique()),
    'label_distribution_unit_level': {c: {s: float(dft[dft.split == s][c].mean())
                                         for s in SPLIT_NAMES} for c in LABEL_COLS},
    'leakage_groups_crossing_splits': int((dft.groupby('group_id')['split'].nunique() > 1).sum()),
}
with open(os.path.join(OUT_DIR, 'splits_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))

print('\nFile di data/processed/:')
for fn in sorted(os.listdir(OUT_DIR)):
    print(f'  {fn:30s} {os.path.getsize(os.path.join(OUT_DIR, fn)):>12,} byte')

{
  "seed": 42,
  "aggregation_method": "majority_safety_first (v/n >= 0.5), semua 6 label",
  "rows_input": 43692,
  "rows_clean": 43520,
  "rows_dropped_minority_content": 171,
  "units_empty_dropped": 1,
  "units_text_recovered": [
    "103-150"
  ],
  "units_final": 28448,
  "groups_final": 26157,
  "label_distribution_unit_level": {
    "toxicity": {
      "train": 0.15340738516961874,
      "val": 0.16032158902813903,
      "test": 0.1582801795416962
    },
    "profanity_obscenity": {
      "train": 0.027219053337336135,
      "val": 0.042563253724284704,
      "test": 0.042523033309709427
    },
    "threat_incitement_to_violence": {
      "train": 0.037376163314320025,
      "val": 0.03783400331047529,
      "test": 0.03803449090479565
    },
    "insults": {
      "train": 0.07760432302611828,
      "val": 0.08394419484511705,
      "test": 0.08362863217576187
    },
    "identity_attack": {
      "train": 0.07625337736415491,
      "val": 0.07850555686923623,
      "test": 0

In [8]:
print('=== VERIFIKASI AKHIR ===')
for s in SPLIT_NAMES:
    p = os.path.join(OUT_DIR, s + '.jsonl')
    with open(p, encoding='utf-8') as f:
        recs = [json.loads(l) for l in f if l.strip()]
    ok_text = all(r['text'].strip() for r in recs)
    ok_keys = all(set(LABEL_COLS) <= set(r) for r in recs)
    print(f'{s:6s}: {len(recs):>6,} baris | text tidak kosong: {ok_text} | label lengkap: {ok_keys}')

print('\nContoh record test.jsonl (pertama):')
with open(os.path.join(OUT_DIR, 'test.jsonl'), encoding='utf-8') as f:
    print(json.dumps(json.loads(next(f)), ensure_ascii=False, indent=2))

=== VERIFIKASI AKHIR ===


train : 19,986 baris | text tidak kosong: True | label lengkap: True
val   :  4,229 baris | text tidak kosong: True | label lengkap: True
test  :  4,233 baris | text tidak kosong: True | label lengkap: True

Contoh record test.jsonl (pertama):
{
  "text_id": "2-8",
  "text": "YENI WAHED agama KRISTEN",
  "initial_paragraph": "",
  "topic": "Kristen, Terpolarisasi",
  "n_annotators": 2,
  "toxicity": 1,
  "agreement_toxicity": 0.5,
  "tie_break_applied_toxicity": true,
  "profanity_obscenity": 0,
  "agreement_profanity_obscenity": 0.0,
  "tie_break_applied_profanity_obscenity": false,
  "threat_incitement_to_violence": 0,
  "agreement_threat_incitement_to_violence": 0.0,
  "tie_break_applied_threat_incitement_to_violence": false,
  "insults": 0,
  "agreement_insults": 0.0,
  "tie_break_applied_insults": false,
  "identity_attack": 1,
  "agreement_identity_attack": 0.5,
  "tie_break_applied_identity_attack": true,
  "sexually_explicit": 0,
  "agreement_sexually_explicit": 0.0,
  "tie_bre

## Ringkasan Pipeline

```
43.692 baris anotasi (raw)
  -> pembersihan konten minoritas (171 baris) & 1 unit kosong total (1 baris)
  -> 43.520 baris valid -> agregasi majority+safety-first per text_id (28.448 unit)
  -> group_id dari konten dinormalisasi (±26.3xx grup)
  -> iterative stratification group-level 70/15/15
  -> re-mapping unit -> train/val/test.jsonl + agreement_metadata.jsonl + splits_summary.json
```

Label training = 6 kolom asli, tie-break safety-first untuk SEMUA label, flag `tie_break_applied_*` tersimpan untuk ablation study.

> Lanjut ke tahap berikutnya (pemodelan IndoBERT vs IndoBERTweet) menunggu instruksi.